# judged: what it does, what a miss costs, what you gain

**Started with `notebooks/run`, everything below is already computed.** The presentation is **`costs.pdf`**
(16:9 slides, link at the bottom):

1. What judged does
2. What one miss costs (the chart)
3. What you gain
4. Why judged
5. Straight answers, for the questions you will get
6. Sources, checked on this run

A last section takes your own incident records; with records, the PDF gets one more slide.
Opened another way and see no output? Menu **Run → Run All Cells**. Charts and PDF need matplotlib.

**Before you present:** read the rows' small print on slide 2. The figures come from different samples, are not
added together, and none of them is a measured saving from judged.

In [ ]:
import html, json, os, re, textwrap, urllib.request
from datetime import date
from pathlib import Path

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("No slides and no PDF: matplotlib is not installed in this Python (pip install matplotlib)")

OFFLINE = False          # True: no downloads at all (rates and source pages are skipped)
CURRENCY = "EUR"         # slide 2 and your records use this currency

DATA = json.loads(Path("costdata.json").read_text())

def download(url, limit=3_000_000, tries=2):
    req = urllib.request.Request(url, headers={"User-Agent": "judged-notebook/0 public list download"})
    for n in range(tries):
        try:
            with urllib.request.urlopen(req, timeout=30) as r:
                return r.read(limit).decode("utf-8", "replace")
        except TimeoutError:
            if n == tries - 1:
                raise

def ecb_rates():
    # {currency: units per 1 EUR}; None when offline or the download fails
    if OFFLINE:
        return None
    try:
        xml = download("https://www.ecb.europa.eu/stats/eurofxref/eurofxref-daily.xml")
        rates = {"EUR": 1.0}
        for cur, rate in re.findall(r"currency='([A-Z]{3})'\s+rate='([0-9.]+)'", xml):
            rates[cur] = float(rate)
        return rates if len(rates) > 1 else None
    except (OSError, ValueError) as e:
        print("no exchange rates:", e)
        return None

def convert(amount, src, dst, rates=None):
    rates = RATES if rates is None else rates
    if src == dst:
        return amount
    if not rates or src not in rates or dst not in rates:
        return None
    return amount / rates[src] * rates[dst]

RATES = ecb_rates()
print("rates:", "downloaded, %d currencies" % len(RATES) if RATES else "none, figures stay in their own currency")

In [ ]:
def sources():
    yield from DATA["figures"].items()
    yield from DATA["gains"].items()
    yield "survey", DATA["survey"]

def page_text(page):
    return re.sub(r"\s+", " ", html.unescape(re.sub(r"<[^>]+>", " ", page)))

def verify(entries, fetch=download):
    # {id: "confirmed" | "NOT FOUND" | "open by hand"}; the last one means this script could not load the page
    out = {}
    for key, entry in entries:
        try:
            page = fetch(entry["url"])
        except (OSError, ValueError):
            out[key] = "open by hand"
            continue
        out[key] = "confirmed" if entry["quote"] in page or entry["quote"] in page_text(page) else "NOT FOUND"
    return out

CHECKS = {} if OFFLINE else verify(sources())
for key, state in CHECKS.items():
    print(f"{state:<14} {key}")
if "NOT FOUND" in CHECKS.values():
    print("\nWARNING: a number is no longer on its page. Do not present it before you have looked.")

In [ ]:
SURFACE, INK, INK2, MUTED, LINE = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e4e3df"
BLUE, BLUE_LIGHT, BLUE_DARK, GREY = "#2a78d6", "#b7d3f6", "#184f95", "#9b9993"
SLIDE = (13.33, 7.5)     # 16:9

def slide(title, subtitle=""):
    page = plt.figure(figsize=SLIDE, facecolor=SURFACE)
    page.add_axes([0, 0, 1, 1], zorder=-1).axis("off")     # Jupyter shows nothing for a figure without axes
    page.text(0.05, 0.915, title, fontsize=26, color=INK, weight="bold", va="center")
    page.text(0.05, 0.845, subtitle, fontsize=13, color=INK2, va="center")
    return page

def words(page, x, y, text, width, size=12, color=INK, **kw):
    page.text(x, y, textwrap.fill(text, width), fontsize=size, color=color, va="top", linespacing=1.35, **kw)

def rule(page, y, x0=0.05, x1=0.95):
    page.add_artist(plt.Line2D([x0, x1], [y, y], color=LINE, linewidth=1, transform=page.transFigure))

def show_slide(page):
    plt.show()
    plt.close(page)

## Slide 1 · What judged does

In [ ]:
WHAT = [
    ("Dates that run out", "TLS certificates, SSH and GPG keys, domains, licences, contracts with a notice period",
     "an alert 30, 14, 7 and 1 days before"),
    ("The computer", "firewall, SSH server, updates, disk encryption, backup evidence, anti-malware",
     "pass, warn or fail, each with the evidence"),
    ("Your code", "secrets left in files, dependencies with known vulnerabilities (also actively exploited), licences",
     "the file and line, or the package and severity"),
    ("Official texts", "GDPR, NIS2, DORA and CRA amendments, Cyber Essentials pages, the Polish data protection act",
     "an alert when a source changes"),
    ("Website and mail", "certificate, security headers, SPF and DMARC against spoofed mail",
     "what is missing, before someone else finds it"),
    ("Audits and certification", "Cyber Essentials, ISO 27001 and GDPR: machine evidence, reviews and certificates with owner and proof, legal deadlines",
     "what applies, what proves it, what is still open"),
]

def slide_what():
    page = slide("judged watches what a small firm cannot afford to miss",
                 "Every check says pass, warn or fail. What cannot wait comes as an alert to your phone, mail or chat.")
    for x, head in ((0.05, "IT WATCHES"), (0.28, "WHAT EXACTLY"), (0.69, "YOU GET")):
        page.text(x, 0.755, head, fontsize=10, color=MUTED, weight="bold")
    y = 0.715
    for name, what, get in WHAT:
        rule(page, y + 0.018)
        words(page, 0.05, y - 0.012, name, 24, 15, INK, weight="bold")
        words(page, 0.28, y - 0.012, what, 56, 12.5, INK2)
        words(page, 0.69, y - 0.012, get, 32, 12.5, BLUE_DARK)
        y -= 0.107
    return page

if plt:
    show_slide(slide_what())

## Slide 2 · What one miss costs

In [ ]:
# (figure id, minutes multiplier, headline, small print, is a legal maximum)
MISSES = [
    ("downtime", 60, "Certificate, key, domain or licence runs out",
     "per HOUR your service is down; small business, 2016 figure\njudged: warns 30, 14, 7 and 1 days before", False),
    ("bec", 1, "Fake invoice or fake-boss e-mail",
     "average loss per report, US 2024\njudged: checks SPF and DMARC, which cover one way in", False),
    ("ransom", 1, "Ransomware",
     "average recovery cost, firms of 100 to 5,000 staff\njudged: checks backup evidence and anti-malware", False),
    ("breach", 1, "Data breach",
     "global average, NOT a small-firm figure\njudged: checks known holes, secrets, host settings", False),
    ("nis2", 1, "NIS2 fine",
     "legal MAXIMUM, essential entities\njudged: only lists the incident reporting deadline", True),
    ("gdpr", 1, "GDPR fine",
     "legal MAXIMUM, serious tier\njudged: gathers machine evidence, lists the breach deadline", True),
]
CONTRACT_LINE = "Not on the chart: a contract renewed because you missed its notice date costs a full term. judged lists those dates."

def short(v):
    if v >= 1e6:
        return f"{v/1e6:,.1f}".removesuffix(".0") + " M"
    return f"{v/1e3:,.1f}".removesuffix(".0") + " k" if v >= 1e3 else f"{v:,.0f}"

def miss_rows(cur=None):
    # [(headline, small print, low, high, currency, legal maximum)]
    cur = cur or CURRENCY
    out = []
    for key, times, head, small, legal in MISSES:
        f = DATA["figures"][key]
        lo, hi = convert(f["low"] * times, f["currency"], cur), convert(f["high"] * times, f["currency"], cur)
        used = cur
        if lo is None:
            lo, hi, used = f["low"] * times, f["high"] * times, f["currency"]
        out.append((head, small, lo, hi, used, legal))
    return out

def slide_costs():
    from matplotlib.patches import Patch
    page = slide("What one miss costs", "Published figures for things judged watches. Separate cases from separate sources: not a total, not a forecast.")
    ax = page.add_axes([0.465, 0.19, 0.45, 0.60], facecolor=SURFACE)
    rows = miss_rows()
    floor = 1e3
    for n, (head, small, lo, hi, cur, legal) in enumerate(rows):
        if legal:
            ax.barh(n, lo - floor, left=floor, height=0.52, color=SURFACE, edgecolor=GREY, hatch="////", linewidth=1)
        else:
            ax.barh(n, lo - floor, left=floor, height=0.52, color=BLUE)
            if hi > lo:
                ax.barh(n, hi - lo, left=lo, height=0.52, color=BLUE_LIGHT)
        label = f"{cur} {short(lo)} to {short(hi)}" if hi > lo else f"{'up to ' if legal else ''}{cur} {short(lo)}"
        ax.text(max(lo, hi) * 1.18, n, label, va="center", fontsize=13, color=INK, weight="bold")
        y = ax.get_position().y1 - (n + 0.5) / len(rows) * ax.get_position().height
        page.text(0.05, y + 0.027, head, fontsize=13.5, color=INK, weight="bold", va="center")
        page.text(0.05, y - 0.018, small, fontsize=9.5, color=INK2, va="center", linespacing=1.4)
    ax.set_ylim(len(rows) - 0.5, -0.5)
    ax.set_xscale("log")
    ax.set_xlim(floor, 6e8)
    ax.set_xticks([1e3, 1e4, 1e5, 1e6, 1e7], ["1 k", "10 k", "100 k", "1 M", "10 M"])
    ax.tick_params(axis="y", left=False, labelleft=False)
    ax.tick_params(axis="x", colors=MUTED, length=0, which="both")
    ax.minorticks_off()
    ax.grid(axis="x", color=LINE)
    ax.set_axisbelow(True)
    for side in ax.spines.values():
        side.set_visible(False)
    page.legend(handles=[Patch(color=BLUE, label="published cost"), Patch(color=BLUE_LIGHT, label="upper end of the range"),
                         Patch(facecolor=SURFACE, edgecolor=GREY, hatch="////", label="legal maximum, not a typical fine")],
                loc="lower right", bbox_to_anchor=(0.95, 0.115), ncols=3, frameon=False, fontsize=10)
    page.text(0.95, 0.098, "Scale: each line is ten times the one before.", fontsize=9, color=MUTED, ha="right")
    rule(page, 0.082)
    page.text(0.05, 0.048, CONTRACT_LINE, fontsize=12.5, color=BLUE_DARK, weight="bold", va="center")
    return page

if plt:
    show_slide(slide_costs())

## Slide 3 · What you gain

In [ ]:
GAINS = [
    ("Be ready to bid", "Public contracts",
     "UK central government: for contracts with higher cyber risk, Cyber Essentials (or equivalent) can be required before "
     "award. Not every contract asks for it.", "contracts"),
    ("up to GBP 25,000", "Cyber insurance cover",
     "UK organisations with turnover under GBP 20m that certify the whole organisation are entitled to cyber liability "
     "insurance with a 24-hour incident helpline. Opting in is a condition.", "insurance"),
    ("92%", "Fewer insurance claims",
     "Organisations with Cyber Essentials are 92% less likely to claim than those without it. Insurer's data comparing two "
     "groups: it is about the certificate, not about judged.", "claims"),
]
GAIN_BRIDGE = ("Where judged comes in: it helps you get to certification and stay ready for renewal. Every Cyber Essentials "
               "requirement listed with what is still open, machine evidence where a check bears on one, certificate dates, "
               "and an alert when the NCSC requirements change. A certification body issues the certificate, from GBP 320 + VAT.")

def slide_gain():
    from matplotlib.patches import FancyBboxPatch
    page = slide("What you gain", "What UK Cyber Essentials certification brings, and how judged helps you get ready for it.")
    for n, (big, name, text, _) in enumerate(GAINS):
        x = 0.05 + n * 0.307
        page.add_artist(FancyBboxPatch((x, 0.31), 0.285, 0.46, boxstyle="round,pad=0,rounding_size=0.012", facecolor="#ffffff",
                                       edgecolor=LINE, transform=page.transFigure))
        page.text(x + 0.02, 0.69, big, fontsize=28 if len(big) < 12 else 21, color=BLUE_DARK, weight="bold", va="center")
        page.text(x + 0.02, 0.605, name, fontsize=14, color=INK, weight="bold", va="center")
        words(page, x + 0.02, 0.555, text, 34, 12, INK2)
    words(page, 0.05, 0.235, GAIN_BRIDGE, 118, 13.5, INK)
    page.text(0.05, 0.045, "Sources: PPN 014 (GOV.UK), IASME, UK National Cyber Security Centre. Links on the last slide.", fontsize=9, color=MUTED)
    return page

if plt:
    show_slide(slide_gain())

## Slide 4 · Why judged

In [ ]:
def pays():
    lo, hi, cur = miss_rows()[0][2:5]
    return [("one hour of downtime that a warning prevented", f"{cur} {short(lo)} to {short(hi)} (small business, 2016 figure)"),
            ("one unwanted contract, cancelled in time", "worth a full term of that contract"),
            ("one weak spot fixed before an attacker finds it", "what that is worth depends on what it protects")]

TRUST = [
    ("Free", "MIT licence, no subscription. Your cost is the time to set it up."),
    ("Hands off", "Checks read and report: nothing is repaired or deleted. Renewal, if you turn it on, runs your own ACME client."),
    ("No false green", "What it could not check is shown as unknown or manual, never as passed."),
    ("Keeps secrets secret", "Secret values are never printed, only the name and the file and line."),
    ("Yours", "Runs on your own machine. Open source: you can read every check."),
]

def slide_why():
    page = slide("Why judged", "It costs nothing to license, and one caught miss can repay the time you put in.")
    page.text(0.05, 0.755, "Any one of these can repay your set-up time", fontsize=15, color=INK, weight="bold")
    y = 0.68
    for text, worth in pays():
        page.add_artist(plt.Rectangle((0.052, y - 0.022), 0.009, 0.016, color=BLUE, transform=page.transFigure))
        words(page, 0.075, y, text, 36, 14, INK)
        words(page, 0.075, y - 0.085, worth, 58, 11, BLUE_DARK)
        y -= 0.175
    page.add_artist(plt.Line2D([0.47, 0.47], [0.12, 0.77], color=LINE, linewidth=1, transform=page.transFigure))
    y = 0.755
    for name, text in TRUST:
        page.text(0.51, y, name, fontsize=15, color=BLUE_DARK, weight="bold")
        words(page, 0.51, y - 0.028, text, 64, 12, INK2)
        y -= 0.135
    return page

if plt:
    show_slide(slide_why())

## Slide 5 · Straight answers

Show it, or keep it for the questions. Saying these first is safer than being caught.

In [ ]:
SV = DATA["survey"]
ANSWERS = [
    ("How much will judged save us?",
     "Nobody has measured it yet. You see what single misses cost; your own incident records give your figure."),
    ("Does a typical breach really cost that much?",
     f"No. UK survey 2025/26: {SV['noticed_percent']}% of businesses noticed a breach or attack, mostly phishing. The median cost of the "
     f"worst one was GBP {SV['median']}; 95% said GBP {SV['p95']:,} or less (medium and large firms: GBP {SV['p95_medium_large']:,}). "
     "The big figures are the costly cases, from other samples."),
    ("Does it get us certified?", "It prepares you: evidence, reviews with owner and proof, and what is still open. The certificate "
     "comes from a certification body, and its assessor decides what is enough."),
    ("Does it work on Windows or Mac?", "The computer checks are Linux only today. Windows and macOS are planned."),
    ("Has the control list been reviewed?", "Not yet, and neither has the calendar of legal deadlines. The tool labels them UNREVIEWED in its own output."),
]

def slide_answers():
    page = slide("Straight answers", "The questions a careful buyer asks.")
    y = 0.76
    for question, answer in ANSWERS:
        rule(page, y + 0.028)
        words(page, 0.05, y, question, 34, 14, INK, weight="bold")
        words(page, 0.38, y, answer, 82, 12.5, INK2)
        y -= 0.138 if len(answer) < 180 else 0.165
    return page

if plt:
    show_slide(slide_answers())

## Your own figure (optional): only what really happened

Write down past misses that a judged check watches, each with a cost you can show: an invoice, a ticket, a bank
statement, a signed contract. No chances, no shares. Set `RECORDS_SINCE` to the day your records start being
complete. An empty list with a start date means a real zero; no start date means no number.

The result is what those misses **cost you**: the most judged could have saved on them, if the warning had been acted on.

In [ ]:
RECORDS_SINCE = None     # "2023-01-01"
INCIDENTS = [
    # {"date": "2025-03-02", "check": "expiry", "what": "certificate expired, shop down 3 h", "cost": 0.0, "proof": "ticket or invoice number"},
]

WATCHED = {
    "expiry": "certificate, key, domain or licence ran out",
    "contract": "contract renewed because the notice date was missed",
    "dependency": "incident through a dependency with a published vulnerability",
    "secret": "incident through a secret left in a file or repository",
    "host": "incident through a host setting judged checks (firewall, SSH, updates, encryption)",
    "mail": "spoofed mail that SPF or DMARC settings let through",
}

def measured(incidents, since, today=None):
    # None = no records yet. Otherwise what the recorded misses cost; "per_year" only with a full year of records.
    if since is None:
        if incidents:
            raise ValueError("set RECORDS_SINCE: without it the incidents cannot be turned into a yearly figure")
        return None
    today = today or date.today()
    start = date.fromisoformat(since)
    if start > today:
        raise ValueError("RECORDS_SINCE is in the future")
    by_check = {}
    for n, i in enumerate(incidents, 1):
        if i.get("check") not in WATCHED:
            raise ValueError(f"incident {n}: check must be one of {', '.join(WATCHED)}")
        if not str(i.get("proof", "")).strip():
            raise ValueError(f"incident {n}: no proof, so it does not count: name the invoice, ticket or statement")
        if not start <= date.fromisoformat(i["date"]) <= today:
            raise ValueError(f"incident {n}: date is outside the period of your records")
        if not i["cost"] >= 0:
            raise ValueError(f"incident {n}: cost must be 0 or more")
        by_check[i["check"]] = by_check.get(i["check"], 0) + i["cost"]
    years = (today - start).days / 365.25
    total = sum(by_check.values())
    return {"since": since, "years": years, "count": len(incidents), "by_check": by_check, "total": total,
            "per_year": total / years if (today - start).days >= 365 else None}

def verdict(m, cur):
    if m is None:
        return "NO NUMBER YET: no records entered. Anything printed here without them would be a guess."
    text = f"{m['count']} recorded misses since {m['since']} ({m['years']:.1f} years) cost {cur} {m['total']:,.0f}"
    if m["per_year"] is None:
        return text + ". Less than a year of records: no yearly figure."
    return text + f", that is {cur} {m['per_year']:,.0f} a year."

MEASURED = measured(INCIDENTS, RECORDS_SINCE)
print(verdict(MEASURED, CURRENCY))
for check, cost in sorted((MEASURED or {}).get("by_check", {}).items(), key=lambda g: -g[1]):
    print(f"  {CURRENCY} {cost:>12,.0f}   {WATCHED[check]}")

## The presentation: `costs.pdf`

In [ ]:
def records_slides(m, per_page=10):
    body = [[i["date"], textwrap.shorten(i["what"], 90, placeholder="..."), i["check"], f"{i['cost']:,.0f}",
             textwrap.shorten(i["proof"], 40, placeholder="...")] for i in INCIDENTS] or [["-", "no incidents in this period", "-", "0", "-"]]
    for start in range(0, len(body), per_page):
        part = [[row[0], textwrap.fill(row[1], 46), row[2], row[3], textwrap.fill(row[4], 24)] for row in body[start:start + per_page]]
        page = slide(f"Your records, {CURRENCY}", "What past misses in judged's areas cost you, each with its proof.")
        ax = page.add_axes([0.05, 0.14, 0.90, 0.64])
        ax.axis("off")
        height = 0.09
        t = ax.table(cellText=part, colLabels=["date", "what happened", "check", "cost", "proof"], colWidths=[0.11, 0.41, 0.12, 0.10, 0.26],
                     cellLoc="left", colLoc="left", bbox=[0, 1 - height * (len(part) + 1), 1, height * (len(part) + 1)])
        t.auto_set_font_size(False)
        t.set_fontsize(10)
        for cell in t.get_celld().values():
            cell.PAD = 0.03
            cell.set_edgecolor(LINE)
            cell.set_facecolor(SURFACE)
        words(page, 0.05, 0.095, verdict(m, CURRENCY), 140, 11, INK)
        yield page

def slide_sources():
    page = slide("Sources", f"Checked by this notebook on {date.today().isoformat()}. The links can be clicked. "
                            "'open by hand': the script could not load the page.")
    y = 0.765
    for key, e in sources():
        rule(page, y + 0.022)
        page.text(0.05, y, e["source"], fontsize=10.5, color=INK, va="center")
        link = e["url"] if len(e["url"]) <= 140 else e["url"][:137] + "..."
        page.text(0.05, y - 0.027, link, fontsize=8.5, color=BLUE_DARK, va="center", url=e["url"])
        page.text(0.95, y, CHECKS.get(key, "not checked"), fontsize=10, color=INK2, ha="right", va="center")
        y -= 0.067
    return page

def slides():
    yield from (slide_what(), slide_costs(), slide_gain(), slide_why(), slide_answers())
    m = measured(INCIDENTS, RECORDS_SINCE)
    if m is not None:
        yield from records_slides(m)
    yield slide_sources()

def report(path):
    from matplotlib.backends.backend_pdf import PdfPages
    count = 0
    with PdfPages(path) as pdf:
        for page in slides():
            pdf.savefig(page, facecolor=page.get_facecolor())
            plt.close(page)
            count += 1
    return count

if plt:
    PDF = Path(os.environ.get("JUDGED_REPORT_DIR", ".")) / "costs.pdf"
    PAGES = report(PDF)
    print(PAGES, "slides")
    try:
        from IPython.display import FileLink, display
        display(FileLink(str(PDF), result_html_prefix="Open the presentation: "))
    except ImportError:
        print("presentation:", PDF)